In [ ]:
# ---- 1. mount + locate the file ----
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
pd.set_option('display.max_columns', None)

In [ ]:
from google.colab import userdata
import os
os.environ["OPENAI_API_KEY"]   = userdata.get("OPENAI_API_KEY")
# os.environ["GEMINI_API_KEY"]   = userdata.get("GEMINI_API_KEY")
os.environ["DEEPSEEK_API_KEY"] = userdata.get("DEEPSEEK_API_KEY")
# print(build_clients().keys())

In [ ]:
!pip install openai pandas pyarrow -q

In [ ]:
#!/usr/bin/env python3
"""
tick_classify_sharded.py  —  Google Colab (multi-notebook safe)
===============================================================
Same classifier as tick_classify.py, with TWO additions:

 A) SHARDING — run N notebooks in parallel WITHOUT cache collisions.
    Each notebook sets SHARD_ID (0..N_SHARDS-1); it processes only its slice
    (deterministic hash of oa_id) and writes to ITS OWN cache file
    tickclass_cache_shard{K}.jsonl. Google Drive does NOT support concurrent
    appends to one file — each notebook keeps its own buffer and the last sync
    wins, which is why your earlier parallel run kept only one notebook's cache.
    Run merge_shards() at the end to combine everything.

 B) LLM_ON_ALL — optionally send the RULE-RESOLVED records to the LLMs too, so
    the regex layer becomes a COMPARATOR, not just a cost saver. You get a
    rule-vs-LLM agreement matrix: which rules are reliable, which systematically
    misfire, and a publishable methods number. Both labels are kept side by side
    (rule_class + llm_class + final class).

CACHE SAFETY: writes to LOCAL disk (fast, reliable) and copies to Drive every
BACKUP_EVERY batches + at the end + in a finally block. Restores from Drive on
start, so a crashed/disconnected session resumes.

SETUP
  !pip install openai pandas pyarrow -q
"""

import hashlib, json, os, re, shutil, sys, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import pandas as pd

# =========================================================================== #
# CONFIG  —  CHANGE SHARD_ID IN EACH NOTEBOOK (0,1,2,3,4)                      #
# =========================================================================== #
SHARD_ID   = 4            # <<<<<< 0 in notebook 1, 1 in notebook 2, ...
N_SHARDS   = 5            # total number of parallel notebooks

IN_PARQUET  = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/with_methods_triples.parquet"
DRIVE_DIR   = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/tickclass"
OUT_PARQUET = DRIVE_DIR + "/with_tickclass.parquet"
VALID_CSV   = DRIVE_DIR + "/tickclass_validation.csv"

LOCAL_CACHE = f"/content/tickclass_cache_shard{SHARD_ID}.jsonl"          # fast, local
DRIVE_CACHE = f"{DRIVE_DIR}/tickclass_cache_shard{SHARD_ID}.jsonl"       # durable copy
BACKUP_EVERY = 10          # batches between Drive backups

BATCH       = 12
WORKERS     = 5
RUN_LLM     = True
LLM_ON_ALL  = True        # True = also LLM the rule-resolved records (comparator)
LLM_ON_ALL_SAMPLE = 0   # if LLM_ON_ALL, cap rule-resolved records sent (0 = all)

Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)

CLASSES = ["tick_organism", "tick_derived", "pathogen_in_tick", "tick_control",
           "host_infestation", "tick_distribution_model", "disease_only",
           "not_tick", "uncertain"]
TICK_EXAMINED_CLASSES = {"tick_organism", "tick_derived", "pathogen_in_tick",
                         "tick_control", "host_infestation"}

try:
    from google.colab import userdata
    _secret = userdata.get
except Exception:
    _secret = os.environ.get


def S(x):
    return "" if x is None or (isinstance(x, float) and pd.isna(x)) or x is pd.NA else str(x)
def short_id(x): return S(x).rsplit("/", 1)[-1]

def in_my_shard(oa_id):
    """Deterministic, stable assignment — same id always goes to the same shard."""
    h = int(hashlib.md5(short_id(oa_id).encode()).hexdigest()[:8], 16)
    return h % N_SHARDS == SHARD_ID


# =========================================================================== #
# rule triage (unchanged logic, exposed so we can compare it to the LLM)      #
# =========================================================================== #
# TICK_GENERA = r"(ixodes|amblyomma|rhipicephalus|hyalomma|dermacentor|haemaphysalis|" \
#               r"boophilus|argas|ornithodoros|otobius|nuttalliella|bothriocroton|carios)"
TICK_GENERA = r"(ixodes|amblyomma|rhipicephalus|rhipicephallus|hyalomma|dermacentor|haemaphysalis|" \
              r"boophilus|argas|ornithodoros|otobius|nuttalliella|bothriocroton|carios|nothoaspis|ixodidae|anomalohimalaya|cosmiomma|margaropus|nosomma|rhipicentor|antricola|ixodida)"
RX = {
    "derived":  re.compile(r"\b(tick (saliva|salivary|midgut|gut|cuticle|haemolymph|hemolymph)|"
                           r"salivary gland|ide8|ise6|bme26|tick cell line|bm86|"
                           r"tick[- ]derived|anti[- ]tick vaccine|tick antigen)\b", re.I),
    "control":  re.compile(r"\b(acaricid|ixodicid|repellent|tick control|amitraz|"
                           r"permethrin|fipronil|resistance to acaricide|dipping)\b", re.I),
    "model":    re.compile(r"\b(species distribution model|habitat suitability|niche model|"
                           r"maxent|risk map|climate (change )?(projection|scenario)|rcp|ssp)\b", re.I),
    "in_tick":  re.compile(r"\b(in (field[- ]collected |questing )?ticks|from ticks|"
                           r"tick[s]? (were |was )?(collected|screened|tested|examined|dissected)|"
                           r"infection (rate|prevalence) in ticks|minimum infection rate)\b", re.I),
    "infest":   re.compile(r"\b(tick (burden|infestation|load|count)|infested (with|by) ticks|"
                           r"tick[- ]bite)\b", re.I),
    "organism": re.compile(r"\b(questing|tick (fauna|survey|abundance|phenology|life cycle|"
                           r"morpholog|taxonom|new species|genome|microbiome)|"
                           r"host[- ]seeking|drag(ging)? sampl)\b", re.I),
    "disease":  re.compile(r"\b(patients?|clinical trial|seroprevalence in (humans|patients)|"
                           r"case report|hospital|serosurvey|doxycycline|vaccine efficacy in)\b", re.I),
    "nottick":  re.compile(r"\b(tick[- ]box|check ?list tick|tick mark|per tick|clock tick)\b", re.I),
}

def rule_triage(rec):
    txt = f"{S(rec.get('oa_title'))} {S(rec.get('oa_abstract'))}"
    ents = S(rec.get("ent_ticks")); mesh = S(rec.get("oa_mesh")).lower()
    has_taxon = bool(re.search(TICK_GENERA, txt, re.I) or ents)
    tick_mesh = any(k in mesh for k in ("ticks", "ixodidae", "argasidae", "tick infestations"))
    if RX["nottick"].search(txt) and not has_taxon and not tick_mesh: return "not_tick", "high"
    if RX["derived"].search(txt): return "tick_derived", "high"
    if RX["control"].search(txt) and (has_taxon or tick_mesh): return "tick_control", "medium"
    if RX["model"].search(txt) and (has_taxon or tick_mesh) and not RX["in_tick"].search(txt):
        return "tick_distribution_model", "medium"
    if RX["in_tick"].search(txt) and re.search(
            r"\b(borrelia|rickettsia|anaplasma|ehrlichia|babesia|theileria|francisella|"
            r"coxiella|virus|pathogen|bacteri)\b", txt, re.I):
        return "pathogen_in_tick", "medium"
    if RX["organism"].search(txt) and has_taxon: return "tick_organism", "medium"
    if RX["infest"].search(txt): return "host_infestation", "medium"
    if RX["disease"].search(txt) and not has_taxon and not RX["in_tick"].search(txt):
        return "disease_only", "medium"
    return None, None


# =========================================================================== #
# LLM prompt + calls (identical taxonomy to the codebook)                     #
# =========================================================================== #
SYSTEM_PROMPT = """You classify biomedical/veterinary abstracts by their OBJECT OF STUDY in tick research. The papers are already known to be tick-relevant; your job is ONLY to say what was actually studied.

Assign EXACTLY ONE primary class. If several apply, the LOWEST NUMBER wins.

1 tick_organism — a tick (any life stage) is the direct object: taxonomy, morphology, genomics, physiology, behaviour, ecology, phenology, field surveys/collection, tick microbiome, tick feeding/xenodiagnosis experiments.
2 tick_derived — material or molecules DERIVED from ticks: saliva, salivary glands, midgut, cuticle, tick cell lines (IDE8, ISE6), tick antigens/anti-tick vaccines (e.g. Bm86), tick-derived proteins.
3 pathogen_in_tick — a pathogen is detected/isolated/characterised IN OR FROM ticks (the tick is the sample matrix): PCR screening of ticks, infection prevalence in ticks, pathogen isolation from ticks.
4 tick_control — interventions targeting ticks: acaricides, resistance, repellents, biological control, integrated tick management, treated-livestock trials measuring tick burden.
5 host_infestation — the host (animal or human) is the unit AND ticks on the host are examined or counted: tick burden on cattle, infestation prevalence in dogs, tick-bite studies where ticks were identified.
6 tick_distribution_model — ticks are the conceptual object but NO ticks were handled: species distribution/niche/suitability models, climate projections, risk maps, distribution meta-analyses.
7 disease_only — tick-borne disease in humans/animals with NO tick examined, derived, or modelled: clinical trials, patient serosurveys, outbreak reports, human diagnostics, pathogen vaccines tested in hosts.
8 not_tick — not about ticks at all (screening false positive; e.g. mites, "tick" as a time unit or checkmark).
9 uncertain — title+abstract genuinely insufficient to decide.

BOUNDARY RULES (follow exactly):
- Pathogen sampled from ticks -> pathogen_in_tick. Only hosts sampled but ticks counted -> host_infestation. Neither -> disease_only.
- Tick cell lines / in vitro tick tissue -> tick_derived (NOT tick_organism).
- Reviews/meta-analyses: classify by the OBJECT reviewed, not the design.
- Vaccine antigen FROM a tick -> tick_derived; vaccine against the PATHOGEN tested in hosts -> disease_only.
- Xenodiagnosis / live tick feeding -> tick_organism.
- Tick-bite case report WITH the tick identified -> host_infestation; without -> disease_only.

ALSO RETURN THESE FACETS:
- tick_examined: true if physical ticks were handled/collected/analysed, else false.
- field_collection: "true" | "false" | "unclear" — were ticks collected from the field (vs lab colony, archived, or none)?
- study_setting: field | laboratory | clinical | modelling | review | mixed
- tick_taxa: array of tick genus/species names explicitly mentioned (may be empty)
- confidence: high | medium | low

NEVER infer beyond the text. If the abstract does not say ticks were collected, use "unclear" for field_collection — absence of evidence is not evidence of absence. If you cannot decide the class, return "uncertain" with confidence "low" rather than guessing.

Return ONLY a JSON array, one object per input, in the SAME order:
[{"class":"...","tick_examined":true,"field_collection":"...","study_setting":"...","tick_taxa":[],"confidence":"..."}]
No prose, no markdown fences."""

def _client(which):
    from openai import OpenAI
    if which == "deepseek":
        return OpenAI(api_key=_secret("DEEPSEEK_API_KEY"),
                      base_url="https://api.deepseek.com"), "deepseek-v4-flash" #"deepseek-chat"
    return OpenAI(api_key=_secret("OPENAI_API_KEY")), "gpt-4o-mini"

def _call(client, model, batch):
    lines = [f'{i}. TITLE: {S(r.get("oa_title"))[:1000]}\nABSTRACT: {S(r.get("oa_abstract"))[:3000]}'
             for i, r in enumerate(batch, 1)]
    user = "Classify each record:\n\n" + "\n\n".join(lines)
    for attempt in range(4):
        try:
            resp = client.chat.completions.create(
                model=model, temperature=0, max_tokens=2500,
                messages=[{"role": "system", "content": SYSTEM_PROMPT},
                          {"role": "user", "content": user}])
            txt = re.sub(r"^```(json)?|```$", "", resp.choices[0].message.content.strip(),
                         flags=re.MULTILINE).strip()
            arr = json.loads(txt)
            if isinstance(arr, list) and len(arr) == len(batch):
                return arr
        except Exception:
            time.sleep(2 ** attempt)
    return [None] * len(batch)

def norm(o):
    if not isinstance(o, dict): return None
    c = S(o.get("class")).strip().lower()
    return {"class": c if c in CLASSES else "uncertain",
            "tick_examined": bool(o.get("tick_examined")),
            "field_collection": S(o.get("field_collection")).lower() or "unclear",
            "study_setting": S(o.get("study_setting")).lower() or "",
            "tick_taxa": o.get("tick_taxa") if isinstance(o.get("tick_taxa"), list) else [],
            "confidence": S(o.get("confidence")).lower() or "low"}


# =========================================================================== #
# cache: local write + periodic Drive backup + restore                        #
# =========================================================================== #
def restore_cache():
    if Path(DRIVE_CACHE).exists() and not Path(LOCAL_CACHE).exists():
        shutil.copy(DRIVE_CACHE, LOCAL_CACHE)
        print(f"[cache] restored from Drive: {DRIVE_CACHE}", file=sys.stderr)
    done = {}
    if Path(LOCAL_CACHE).exists():
        for line in open(LOCAL_CACHE):
            try:
                d = json.loads(line); done[d["oa_id"]] = d
            except Exception: pass
    print(f"[cache] shard {SHARD_ID}: {len(done):,} already classified", file=sys.stderr)
    return done

def backup_cache():
    try:
        shutil.copy(LOCAL_CACHE, DRIVE_CACHE)
    except Exception as e:
        print(f"[cache] backup failed: {e}", file=sys.stderr)


# =========================================================================== #
# main LLM loop for this shard                                                #
# =========================================================================== #
def run_llm(records, cache, tag):
    oa_c, oa_m = _client("openai"); ds_c, ds_m = _client("deepseek")
    todo = [r for r in records if short_id(r.get("oa_id")) not in cache]
    if not todo:
        print(f"[llm] {tag}: nothing to do", file=sys.stderr); return cache
    print(f"[llm] {tag}: {len(todo):,} records -> dual-model", file=sys.stderr)
    batches = [todo[i:i + BATCH] for i in range(0, len(todo), BATCH)]
    fh = open(LOCAL_CACHE, "a")

    def work(batch):
        a = _call(oa_c, oa_m, batch); b = _call(ds_c, ds_m, batch)
        out = []
        for rec, ra, rb in zip(batch, a, b):
            na, nb = norm(ra), norm(rb)
            oid = short_id(rec.get("oa_id"))
            rule_cls, _ = rule_triage(rec)
            base = {"oa_id": oid, "rule_class": rule_cls or "",
                    "openai": (na or {}).get("class", ""), "deepseek": (nb or {}).get("class", "")}
            if na and nb and na["class"] == nb["class"]:
                base.update(na); base["method"] = "llm_consensus"
            elif na and nb:
                base.update({"class": "uncertain", "tick_examined": na["tick_examined"] and nb["tick_examined"],
                             "field_collection": "unclear", "study_setting": na["study_setting"],
                             "tick_taxa": na["tick_taxa"], "confidence": "low",
                             "method": "llm_disagree"})
            else:
                one = na or nb or {"class": "uncertain", "tick_examined": False,
                                   "field_collection": "unclear", "study_setting": "",
                                   "tick_taxa": [], "confidence": "low"}
                base.update(one); base["method"] = "llm_single"
            out.append(base)
        return out

    done = 0
    try:
        with ThreadPoolExecutor(max_workers=WORKERS) as ex:
            futs = [ex.submit(work, b) for b in batches]
            for fut in as_completed(futs):
                for d in fut.result():
                    fh.write(json.dumps(d, ensure_ascii=False) + "\n")
                    cache[d["oa_id"]] = d
                fh.flush(); os.fsync(fh.fileno())
                done += 1
                if done % BACKUP_EVERY == 0:
                    backup_cache()
                    print(f"[llm] {tag}: {done:,}/{len(batches):,} batches "
                          f"(cache backed up)", file=sys.stderr)
    finally:
        fh.close(); backup_cache()      # always back up, even on crash/interrupt
    return cache


# =========================================================================== #
# MERGE — run ONCE, in any notebook, after all shards finish                  #
# =========================================================================== #
def merge_shards(n_shards=N_SHARDS):
    """Combine every shard cache into the final classified parquet + report
    rule-vs-LLM agreement."""
    df = pd.read_parquet(IN_PARQUET)
    merged = {}
    for k in range(n_shards):
        p = f"{DRIVE_DIR}/tickclass_cache_shard{k}.jsonl"
        if not Path(p).exists():
            print(f"[merge] shard {k}: MISSING ({p})", file=sys.stderr); continue
        n = 0
        for line in open(p):
            try:
                d = json.loads(line); merged[d["oa_id"]] = d; n += 1
            except Exception: pass
        print(f"[merge] shard {k}: {n:,} records", file=sys.stderr)
    print(f"[merge] total unique: {len(merged):,}", file=sys.stderr)

    rows = []
    for _, r in df.iterrows():
        oid = short_id(r.get("oa_id"))
        d = merged.get(oid)
        rule_cls, rule_conf = rule_triage(r)
        if d:
            rows.append({"oa_id": oid, "tick_object_type": d.get("class", "uncertain"),
                         "rule_class": d.get("rule_class") or rule_cls or "",
                         "llm_openai": d.get("openai", ""), "llm_deepseek": d.get("deepseek", ""),
                         "tick_examined": bool(d.get("tick_examined")),
                         "field_collection": d.get("field_collection", "unclear"),
                         "study_setting": d.get("study_setting", ""),
                         "tick_taxa_llm": ";".join(d.get("tick_taxa") or []),
                         "class_confidence": d.get("confidence", "low"),
                         "class_method": d.get("method", "")})
        else:   # not LLM'd -> fall back to the rule label
            rows.append({"oa_id": oid, "tick_object_type": rule_cls or "uncertain",
                         "rule_class": rule_cls or "", "llm_openai": "", "llm_deepseek": "",
                         "tick_examined": (rule_cls in TICK_EXAMINED_CLASSES) if rule_cls else False,
                         "field_collection": "unclear", "study_setting": "",
                         "tick_taxa_llm": "", "class_confidence": rule_conf or "low",
                         "class_method": "rule"})
    cls = pd.DataFrame(rows)
    out = df.merge(cls, on="oa_id", how="left")
    out.to_parquet(OUT_PARQUET, index=False)
    print(f"[write] {OUT_PARQUET}", file=sys.stderr)

    print("\n===== CLASS DISTRIBUTION =====")
    print(out["tick_object_type"].value_counts().to_string())
    print(f"\ntick_examined = true: {out['tick_examined'].sum():,} ({out['tick_examined'].mean():.1%})")
    print("\n[by method]\n" + out["class_method"].value_counts().to_string())

    # ---- RULE vs LLM agreement (the comparator you asked for) ----
    both = out[(out["rule_class"] != "") & (out["class_method"].str.startswith("llm"))]
    if len(both):
        agree = (both["rule_class"] == both["tick_object_type"]).mean()
        print(f"\n===== RULE vs LLM =====\nrecords with both labels: {len(both):,}")
        print(f"agreement: {agree:.1%}")
        print("\nconfusion (rows = rule, cols = LLM consensus):")
        print(pd.crosstab(both["rule_class"], both["tick_object_type"]).to_string())
        print("\n[read] low agreement for a rule = that regex is unreliable; "
              "inspect and fix it, or drop it and let the LLM decide.")

    # ---- inter-model agreement (reliability statistic for Methods) ----
    llm = out[out["llm_openai"].ne("") & out["llm_deepseek"].ne("")]
    if len(llm):
        print(f"\n[llm reliability] OpenAI vs DeepSeek agreement: "
              f"{(llm['llm_openai'] == llm['llm_deepseek']).mean():.1%} (n={len(llm):,})")
    return out


# =========================================================================== #
if __name__ == "__main__":
    df_all = pd.read_parquet(IN_PARQUET)
    df = df_all[df_all["llm_label"].notna()]

    # df = df.iloc[1:20000] # Slice the DataFrame to process rows 10000 to 19999

    mine = df[df["oa_id"].map(in_my_shard)]
    print(f"[shard {SHARD_ID}/{N_SHARDS}] {len(mine):,} of {len(df):,} records "
          f"({len(mine)/len(df):.1%})", file=sys.stderr)

    cache = restore_cache()
    rule_only, ambiguous = [], []
    for _, r in mine.iterrows():
        cls, _ = rule_triage(r)
        (rule_only if cls else ambiguous).append(r)
    print(f"[triage] rules resolved {len(rule_only):,}; ambiguous {len(ambiguous):,}",
          file=sys.stderr)

    if RUN_LLM:
        run_llm(ambiguous, cache, "ambiguous")
        if LLM_ON_ALL and rule_only:
            sample = rule_only[:LLM_ON_ALL_SAMPLE] if LLM_ON_ALL_SAMPLE else rule_only
            print(f"[compare] also sending {len(sample):,} rule-resolved records "
                  f"to the LLMs for rule-vs-LLM comparison", file=sys.stderr)
            run_llm(sample, cache, "rule-comparison")

    backup_cache()
    print(f"\n[done] shard {SHARD_ID} complete. When ALL shards are done, run in ONE "
          f"notebook:\n    import tick_classify_sharded as t; t.merge_shards()", file=sys.stderr)